# 05 · EDA step 4: skill vs. luck

**Question:** which team stats are real, repeatable qualities of a team, and which are
mostly luck?

**Why this matters for the model.** A pre-game feature only helps if what a team did
before tells us what it will do next. A stat can go with winning inside a game (step 3)
and still be useless for prediction if it bounces around randomly from game to game. This
step decides:
- **which stats get weight in the model**: repeatable stats are trusted, luck-heavy ones are
  dropped or used only over long windows;
- **how long the window should be**: a noisy stat needs many games before it says anything,
  so "last 3 games" of it is mostly noise.

**Method: split-half reliability.**
1. For each team, compute every stat twice: once over its odd-numbered games, once over its
   even-numbered games. Alternating games keeps both halves spread over the same weeks, so
   changes over the season don't matter.
2. Correlate the two halves across the 63 teams.
   - If a stat is mostly skill, teams that are high in one half are high in the other.
   - If it's mostly luck, the halves don't line up.
3. **Spearman-Brown** turns that half-season correlation into:
   - the reliability of a full season of games;
   - **the number of games needed for the stat to be half signal, half noise.** That's the
     guide for window length.

**Check over time:** the same correlation between each team's first and second half of the
season. That captures teams actually changing, which odd/even deliberately hides.

**Setup:** D-I games only, ties included (they're games played), 2025-26. Only team-level
summaries are shown.

In [ ]:
%run ./01_pregame_features.ipynb

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def query(conn, sql, params):
    with conn.cursor() as cur:
        cur.execute(sql, params)
        return pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])


EXTRA_SQL = """
SELECT t.game_id, t.team_id, t.sh_goals, t.en_goals, t.pim, t.shots_p1, t.shots_p2, t.shots_p3
FROM hockey.team_game_stats t JOIN hockey.games g USING (game_id)
WHERE g.season = %s AND g.is_played
"""
GOALS_SQL = """
SELECT go.game_id, go.goal_number, go.team_id, go.period
FROM hockey.goals go JOIN hockey.games g USING (game_id)
WHERE g.season = %s AND g.is_played
"""
with psycopg.connect(settings.database_url) as conn:
    extra = query(conn, EXTRA_SQL, (SEASON,))
    goals = query(conn, GOALS_SQL, (SEASON,))

new_cols = ["game_id", "team_id"] + [c for c in extra.columns if c not in team_games.columns]
tg = team_games.merge(extra[new_cols], on=["game_id", "team_id"], how="left", validate="one_to_one")
goals = goals[goals["game_id"].isin(tg["game_id"])]
print(f"{tg['team_id'].nunique()} teams, {len(tg)} team-games")

## Per-game ingredients

Everything is built from counts, so each half's stats are proper rates (total goals ÷ total
shots, and so on), not averages of per-game percentages.

- **Close-game shots** (new in step 3): shots in periods that *started* tied or within one
  goal. Period 1 always counts.
- **Scored first:** whether the team scored the game's first goal (0-0 games don't count).
- **One-goal games:** the result in games decided by exactly one goal, overtime included.
- **3rd-period goal differential:** goals for minus against in the 3rd period.

In [ ]:
# Opponent's period shots next to each team-game
opp = tg.set_index(["game_id", "team_id"])[["shots_p1", "shots_p2", "shots_p3"]]
o = opp.reindex(pd.MultiIndex.from_frame(tg[["game_id", "opponent_team_id"]])).to_numpy()
tg[["opp_p1", "opp_p2", "opp_p3"]] = o

# Goals by team and period, and each team's margin at the start of each period
g = goals.merge(tg[["game_id", "team_id"]].assign(mine=True), on=["game_id", "team_id"], how="left")
per = goals.groupby(["game_id", "team_id", "period"]).size().rename("n").reset_index()


def goals_in(period_max, period_min=1):
    sel = per[(per["period"] >= period_min) & (per["period"] <= period_max)]
    s = sel.groupby(["game_id", "team_id"])["n"].sum()
    mine = s.reindex(pd.MultiIndex.from_frame(tg[["game_id", "team_id"]]), fill_value=0).to_numpy()
    theirs = s.reindex(pd.MultiIndex.from_frame(tg[["game_id", "opponent_team_id"]]), fill_value=0).to_numpy()
    return mine, theirs


m1, t1 = goals_in(1)
m2, t2 = goals_in(2)
p3f, p3a = goals_in(3, 3)
close_p2 = np.abs(m1 - t1) <= 1  # 2nd period started within one goal
close_p3 = np.abs(m2 - t2) <= 1
tg["close_sf"] = tg["shots_p1"] + tg["shots_p2"] * close_p2 + tg["shots_p3"] * close_p3
tg["close_sa"] = tg["opp_p1"] + tg["opp_p2"] * close_p2 + tg["opp_p3"] * close_p3
tg["p3_gf"], tg["p3_ga"] = p3f, p3a

first = goals.sort_values("goal_number").groupby("game_id")["team_id"].first()
tg["first_goal_game"] = tg["game_id"].isin(first.index).astype(int)
tg["scored_first"] = (tg["game_id"].map(first) == tg["team_id"]).astype(int)

margin = tg["goals_for"] - tg["goals_against"]
tg["one_goal_game"] = (margin.abs() == 1).astype(int)
tg["one_goal_win"] = (margin == 1).astype(int)
tg["win_pts"] = (tg["result"] == "W") + 0.5 * (tg["result"] == "T")
tg["games"] = 1
# Each team's D-I games numbered in date order (1, 2, 3, ...)
tg = tg.sort_values(["team_id", "game_date", "game_id"]).reset_index(drop=True)
tg["game_number"] = tg.groupby("team_id").cumcount() + 1
assert (tg["close_sf"] <= tg["shots_for"]).all() and (tg["close_sa"] <= tg["shots_against"]).all()

## Stats

Each is computed from summed counts over a set of games.

In [ ]:
def r(num, den):
    return num / den.where(den > 0)


def team_stats(t: pd.DataFrame) -> pd.DataFrame:
    """t = summed counts per team over some set of games."""
    sh_pct = r(t["goals_for"] - t["en_goals"], t["shots_for"])
    sv_pct = 1 - r(t["goals_against"], t["shots_against"])
    pp = r(t["pp_goals"], t["pp_opportunities"])
    pk = 1 - r(t["pp_goals_against"], t["times_shorthanded"])
    return pd.DataFrame({
        "win %": r(t["win_pts"], t["games"]),
        "goal diff / game": r(t["goals_for"] - t["goals_against"], t["games"]),
        "shot share": r(t["shots_for"], t["shots_for"] + t["shots_against"]),
        "close-game shot share": r(t["close_sf"], t["close_sf"] + t["close_sa"]),
        "shots for / game": r(t["shots_for"], t["games"]),
        "shots against / game": r(t["shots_against"], t["games"]),
        "shooting %": sh_pct,
        "save %": sv_pct,
        "PDO (sh% + sv%)": sh_pct + sv_pct,
        "PP%": pp,
        "PK%": pk,
        "special teams index (PP% + PK%)": pp + pk,
        "PP chances / game": r(t["pp_opportunities"], t["games"]),
        "times shorthanded / game": r(t["times_shorthanded"], t["games"]),
        "PIM / game": r(t["pim"], t["games"]),
        "% of games scoring first": r(t["scored_first"], t["first_goal_game"]),
        "one-goal game win %": r(t["one_goal_win"], t["one_goal_game"]),
        "3rd-period goal diff / game": r(t["p3_gf"] - t["p3_ga"], t["games"]),
    })


COUNT_COLS = ["games", "win_pts", "goals_for", "goals_against", "en_goals", "shots_for",
              "shots_against", "close_sf", "close_sa", "pp_goals", "pp_opportunities",
              "pp_goals_against", "times_shorthanded", "pim", "scored_first",
              "first_goal_game", "one_goal_game", "one_goal_win", "p3_gf", "p3_ga"]


def by_split(mask):
    a = team_stats(tg[mask].groupby("team_id")[COUNT_COLS].sum())
    b = team_stats(tg[~mask].groupby("team_id")[COUNT_COLS].sum())
    return a, b.loc[a.index]

## Split-half reliability

- **r (odd vs even):** correlation between the two halves across teams, with a 95% interval.
- **full-season reliability:** the share of the differences between teams in a full season
  of this stat that's real (Spearman-Brown). For example, 0.80 means 80% skill and 20% noise.
- **games for 50/50:** how many games until the stat is half signal, half noise. Below 5,
  a short window is fine. Above 30, even a full season is mostly noise.
- **r (1st vs 2nd half):** the same check across time, which also picks up teams changing.

In [ ]:
def fisher_ci(rho, n, z=1.96):
    f = np.arctanh(np.clip(rho, -0.999, 0.999))
    se = 1 / np.sqrt(n - 3)
    return np.tanh(f - z * se), np.tanh(f + z * se)


def games_for_half(r_half, n_half):
    """Spearman-Brown: games m at which reliability = 0.5, from the split-half r."""
    if r_half <= 0:
        return np.inf
    per_game = r_half / (n_half - (n_half - 1) * r_half)  # reliability of one game
    return (1 - per_game) / per_game


odd = tg["game_number"] % 2 == 1
a, b = by_split(odd)
n_teams = len(a)
n_half = tg.groupby("team_id").size().median() / 2

early = tg["game_number"] <= tg.groupby("team_id")["game_number"].transform("max") / 2
e1, e2 = by_split(early)

rel = pd.DataFrame(index=a.columns)
rel["r (odd vs even)"] = [a[c].corr(b[c]) for c in a.columns]
ci = rel["r (odd vs even)"].apply(lambda x: fisher_ci(x, n_teams))
rel["95% low"] = [lo for lo, _ in ci]
rel["95% high"] = [hi for _, hi in ci]
rel["full-season reliability"] = (2 * rel["r (odd vs even)"] / (1 + rel["r (odd vs even)"])).clip(lower=0)
rel["games for 50/50"] = rel["r (odd vs even)"].apply(lambda x: games_for_half(x, n_half))
rel["r (1st vs 2nd half)"] = [e1[c].corr(e2[c]) for c in a.columns]
rel = rel.sort_values("r (odd vs even)", ascending=False)
print(f"{n_teams} teams, about {n_half:.0f} games per half")
rel.style.format({
    "r (odd vs even)": "{:.2f}", "95% low": "{:.2f}", "95% high": "{:.2f}",
    "full-season reliability": "{:.2f}", "games for 50/50": "{:.0f}", "r (1st vs 2nd half)": "{:.2f}",
})

In [ ]:
BLUE, MUTED, GRID, INK = "#2a78d6", "#6b6a66", "#e1e0d9", "#1f1f1d"
fig, ax = plt.subplots(figsize=(7.5, 6))
y = np.arange(len(rel))[::-1]
ax.hlines(y, rel["95% low"], rel["95% high"], color=BLUE, lw=2, alpha=0.35)
ax.plot(rel["r (odd vs even)"], y, "o", color=BLUE, ms=7)
ax.axvline(0, color=MUTED, lw=0.8)
ax.set_yticks(y, rel.index)
ax.set_xlim(-0.6, 1)
ax.set_xlabel("correlation between a team's odd and even games (95% interval)", color=INK)
ax.set_title("Which stats repeat? Higher = more skill, near 0 = mostly luck", loc="left", fontsize=11)
ax.grid(axis="x", color=GRID, lw=0.6)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()

## Check: is the noise just small samples?

Shooting % and save % are rates over a few hundred shots per half. Binomial sampling noise
alone sets how much teams *would* differ if they were all identical. Compare that with how
much they actually differ:
- **true spread ≈ 0:** the differences between teams are what pure chance produces.
- **true spread > 0:** some of the differences are real.

In [ ]:
def true_spread(num, den):
    p = num / den
    observed_var = p.var(ddof=1)
    pooled = num.sum() / den.sum()
    noise_var = (pooled * (1 - pooled) / den).mean()
    return pd.Series({"observed SD": np.sqrt(observed_var), "chance-only SD": np.sqrt(noise_var),
                      "true spread (SD)": np.sqrt(max(observed_var - noise_var, 0)),
                      "share of spread that's real": max(observed_var - noise_var, 0) / observed_var})


season = tg.groupby("team_id")[COUNT_COLS].sum()
spread = pd.DataFrame({
    "save %": true_spread(season["shots_against"] - season["goals_against"], season["shots_against"]),
    "shooting %": true_spread(season["goals_for"] - season["en_goals"], season["shots_for"]),
    "PP%": true_spread(season["pp_goals"], season["pp_opportunities"]),
    "PK%": true_spread(season["times_shorthanded"] - season["pp_goals_against"], season["times_shorthanded"]),
    "shot share": true_spread(season["shots_for"], season["shots_for"] + season["shots_against"]),
}).T
spread.style.format({"observed SD": "{:.3f}", "chance-only SD": "{:.3f}", "true spread (SD)": "{:.3f}",
                     "share of spread that's real": "{:.0%}"})

## Implications for the model

**The stats fall into three tiers.** With 63 teams the intervals are about ±0.2, so neighbors
in the ranking aren't really different. The tiers are clear, though.

**Tier 1: skill, and readable within a handful of games.** These are the core of the model.
- Shot share, shots against, close-game shot share, shots for and goal differential.
- Odd/even r = 0.70–0.82. They reach half signal within **4–8 games**, and hold up from the
  first half of the season to the second (r ≈ 0.65–0.73).
- Shot share is 95% real differences between teams, almost nothing is chance.
- **For the model:** these can be used over short windows (last 5 or 10), and step 5 should
  test whether recent form beats the season figure for them.
- Win % (11 games) is a little noisier than goal differential, which says the same thing
  with less noise.

**Tier 2: partly skill, needs a full season.**
- Special teams (PP%, PK%, their index, PP chances, times shorthanded), save %, PDO, PIM, %
  scoring first and 3rd-period goal differential.
- Odd/even r = 0.30–0.47. Each needs **20–40 games** to reach half signal, which is close to
  a whole season. Only about 40–55% of the spread between teams in PP%, PK% and save % is
  real; the rest is what chance alone produces.
- **For the model:** use them only as **season-to-date** values, never "last 3" or "last 5"
  (those would be mostly noise). Expect them to add little beyond tier 1, and let the
  model's regularization discount them.
- **Special teams in particular:** step 3 showed they decide individual games. Here they
  barely repeat, and PK% hardly carries from the first half of the season to the second
  (r = 0.16). Special teams swing games, but **which team wins the special teams battle is
  hard to know in advance.** The combined index (0.43) is steadier than PP% or PK% alone,
  so it's the one to test first.
- **Save %:** about half real. A goalie-level version (the likely starter's season save %)
  may be steadier than the team's, so it stays on the list for step 5.
- **% scoring first** (0.43) repeats far less than goal differential (0.70). It's a noisier
  copy of team strength and is expected to drop out.

**Tier 3: luck. Drop these from the model.**
- **Shooting %:** r = 0.25. It would take about 54 games to reach half signal, more than a
  season.
- **One-goal game win %:** r = 0.08, indistinguishable from zero. A team's record in close
  games tells you nothing about its next close game.
- **PDO** (shooting % + save %) is only as steady as its save % half, so it's also dropped.
  Save % is tested on its own.

**What this means for windows in step 5.** Test every window only for tier 1. For tier 2,
test season-to-date only. That cuts the number of feature × window combinations roughly in
half, which also lowers the risk of picking a feature that won by luck.

**Close-game shot share** repeats as well as raw shot share (0.81 vs 0.82), so repeatability
doesn't separate them. Step 5 decides on predictive power.